# Notebook 02 — PPO-CF: the counterfactual oracle inside the training loop

Notebook 01 established plain PPO. This one adds the **exact** one-step
counterfactual advantage, computed from the simulator over all K actions, and
compares it against plain PPO under otherwise identical hyperparameters.

$$Q_{CF}(s,a) = r(s,a) + \gamma V_\phi(s'_a)(1-\text{term}_a), \qquad
A_{CF}(s,a) = Q_{CF}(s,a) - \sum_b \pi_{old}(b|s) Q_{CF}(s,b)$$

$$\mathcal{L} = -\sum_a \pi_{old}(a|s)\,
\min\!\big(\rho_a \hat A_{CF}(s,a),\ \mathrm{clip}(\rho_a, 1\pm\epsilon)\hat A_{CF}(s,a)\big),
\qquad \rho_a = \frac{\pi_\theta(a|s)}{\pi_{old}(a|s)}$$

Unclipped this equals $-\sum_a \pi_\theta(a|s) \hat A_{CF}(s,a)$: **every** action
contributes to the gradient weighted by its probability, so the variance from
action *sampling* is eliminated. That is the point of the oracle. Substituting
$A_{CF}(s, a_{taken})$ for the GAE advantage instead would just be GAE with
$\lambda=0$ — high bias, and probably worse.

**Why run the exact oracle at all**, given it needs a simulator and is ~4×
slower? Because it is the *ceiling*. If the exact counterfactual advantage does
not beat GAE, no learned approximation of it can, and the rest of the plan
(NB05's learned $B(s)$, NB06) is not worth building.

The critic is trained on GAE returns in **both** arms. Only the policy loss
changes, so $V_\phi$ stays a value function for $\pi$ — which is what $Q_{CF}$
bootstraps through.

> **Section 1 runs the oracle checks. Do not skip to training.** A broken state
> restore produces $A_{CF}$ values that look completely reasonable and are
> wrong, and nothing downstream can detect it.

---
## ▶ Knobs

In [ ]:
# ----------------------------------------------------------------------------
# EDIT ME
# ----------------------------------------------------------------------------
ENV_CONFIG    = "doorkey5x5_cf"   # doorkey5x5_cf | doorkey6x6_cf | doorkey8x8_cf
SEEDS         = None              # None -> use the config's seeds
FORCE_RETRAIN = False

RUN_ORACLE_CHECKS = True          # section 1. Cheap. Run it.
RUN_CONTROL       = True          # section 2, arm A: plain PPO (pg_mode = gae)
RUN_CF            = True          # section 3, arm B: PPO-CF (pg_mode = cf_all_action)

# The oracle checks validate against a POLICY. A trained one makes the sanity
# probes meaningful (a random policy's critic is noise, so A_CF is too).
# Points at notebook 01's plain 5x5 run; set to None to use a fresh model.
ORACLE_CHECK_RUN  = "doorkey5x5"
ORACLE_CHECK_CKPT = 1.00

OVERRIDES = {
    # "ppo.cf_restore": "fast",     # ~2.5x quicker; section 1.5 verifies it is identical
    # "ppo.total_timesteps": 100_000,
}
# ----------------------------------------------------------------------------

In [ ]:
import sys, pathlib, time

ROOT = pathlib.Path.cwd()
if not (ROOT / "config").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import make_config, seed_dir, RUNS_DIR, FIGURES_DIR
from dataio import load_trajectories, load_checkpoint, list_checkpoints
from utils.logging import read_scalars
from utils.plotting import plot_subgoal_ladder, savefig
from scripts.train import run_seeds

base = make_config(ENV_CONFIG, **OVERRIDES)
if SEEDS is not None:
    base = make_config(ENV_CONFIG, **{**OVERRIDES, "run.seeds": tuple(SEEDS)})
SEEDS = tuple(base.run.seeds)
STEM  = base.run.run_name

# ONE config, two arms. Every hyperparameter is shared by construction; the only
# difference between the runs is the policy gradient.
ARMS = {
    "gae": make_config(ENV_CONFIG, **{**OVERRIDES, "run.seeds": SEEDS,
                                      "ppo.pg_mode": "gae",
                                      "run.run_name": f"{STEM}_gae"}),
    "cf":  make_config(ENV_CONFIG, **{**OVERRIDES, "run.seeds": SEEDS,
                                      "ppo.pg_mode": "cf_all_action",
                                      "run.run_name": f"{STEM}_cf"}),
}
FIG = FIGURES_DIR / f"nb02_{STEM}"
FIG.mkdir(parents=True, exist_ok=True)

print(ARMS["cf"].summary())
print(f"\narms: {[c.run.run_name for c in ARMS.values()]}   seeds: {list(SEEDS)}")

# Measured on DoorKey-5x5, K=7: gae ~2,050 steps/s, cf ~465 ("exact") / ~1,150 ("fast")
rate = {"gae": 2050, "cf": 465 if base.ppo.cf_restore == "exact" else 1150}
print("\nestimated runtime")
for k, c in ARMS.items():
    m = c.ppo.total_timesteps / rate[k] / 60
    print(f"  {k:3s}  {m:5.1f} min/seed  x {len(SEEDS)} seeds = {m*len(SEEDS):5.1f} min")

---
# 1. Test the oracle

Five checks. The first is the one that matters; the rest catch subtler things.

In [ ]:
from envs.env_pool import set_sim_state
from oracle.online import (OnlineOracle, check_replay, check_centering,
                           check_restore_equivalence, landscape_summary)

cfg = ARMS["cf"]
ENV_KW = {"fully_observable": cfg.env.fully_observable} if cfg.env.env_id.startswith("MiniGrid") else {}
K = 7 if cfg.env.env_id.startswith("MiniGrid") else None

# States and a policy to check against.
if ORACLE_CHECK_RUN and (seed_dir(ORACLE_CHECK_RUN, SEEDS[0]) / "trajectories.npz").exists():
    sd   = seed_dir(ORACLE_CHECK_RUN, SEEDS[0])
    traj = load_trajectories(sd / "trajectories.npz")
    ck   = load_checkpoint([p for p in list_checkpoints(sd / "checkpoints")
                            if abs(load_checkpoint(p).fraction - ORACLE_CHECK_CKPT) < 1e-6][0])
    value_fn, probs_fn = ck.values, ck.probs
    src = f"{ORACLE_CHECK_RUN} @ {ORACLE_CHECK_CKPT:.0%}"
    N = min(600, len(traj))
    sims  = traj.sim_state[:N]
    acts  = traj.action[:N]
    rews  = traj.reward[:N]
    nsims = traj.next_sim_state[:N]
    K = traj.n_actions
else:
    from agents.ppo import PPOTrainer
    _t = PPOTrainer(cfg, seed=SEEDS[0], progress=False)
    _t.pool.reset(); _t.collect_rollout()
    T, NE = cfg.ppo.n_steps, cfg.env.n_envs
    f = lambda a: a[:T].reshape((T*NE,) + a.shape[2:])
    sims, acts, rews, nsims = f(_t.buffer.sim_state), f(_t.buffer.actions), f(_t.buffer.rewards), f(_t.buffer.next_sim_state)
    value_fn, probs_fn = _t._values_np, lambda o: _t.model.action_probs(
        __import__("torch").as_tensor(_t._scale(o), dtype=__import__("torch").float32)).numpy()
    K = _t.pool.n_actions
    src = "a fresh, untrained model (sanity probes below will be meaningless)"

print(f"checking against: {src}")
print(f"states: {len(sims)},  actions: {K},  gamma: {cfg.ppo.gamma}")

oracle = OnlineOracle(cfg.env.env_id, K, cfg.ppo.gamma, ENV_KW,
                      cfg.env.max_episode_steps, restore=cfg.ppo.cf_restore)

### 1.1 Restore and replay must be exact

Restore each recorded simulator state, replay the action that was actually
taken, and compare against what the run recorded. **This is the check that
matters.** If it fails, every $A_{CF}$ in the project is meaningless and every
downstream number will look plausible and be wrong.

In [ ]:
rep = check_replay(oracle, sims[:400], acts[:400], rews[:400], nsims[:400])
print(f"  transitions replayed  {rep['n']}")
print(f"  max reward error      {rep['max_reward_error']:.3e}")
print(f"  max state error       {rep['max_state_error']:.3e}")
print(f"  -> {'EXACT' if rep['exact'] else 'MISMATCH — STOP, do not train'}")

### 1.2 Step count must be restored, not zeroed

`oracle/counterfactual.py` (the frozen-checkpoint oracle for NB02) restores with
`elapsed_steps=0`. On MountainCar that is harmless — every reward is −1. On
MiniGrid it is **not**: DoorKey pays `1 − 0.9·(step_count/max_steps)` on success,
so zeroing the step count makes a counterfactual success at step 200 look worth
1.0 instead of 0.72, and it also disables MiniGrid's internal truncation.

`oracle/online.py` restores the state's own step count. The cell below measures
what the difference actually is on your data.

In [ ]:
late = sims[np.argsort(sims[:, -1])[-300:]]          # the states with the highest step_count
t_ok = oracle.transitions(late)
r_bad = np.zeros_like(t_ok["reward"])
for m, s in enumerate(late):
    for a in range(K):
        set_sim_state(oracle.env, s, elapsed_steps=0)      # the NB02 behaviour
        _o, r, _t, _tr, _ = oracle.env.step(a)
        r_bad[m, a] = r

succ = t_ok["terminated"]
print(f"  step_count range in these states : {late[:,-1].min():.0f} - {late[:,-1].max():.0f}")
print(f"  counterfactual successes among them: {int(succ.sum())}")
if succ.sum():
    ok, bad = t_ok['reward'][succ].mean(), r_bad[succ].mean()
    print(f"  mean reward, step_count restored : {ok:.4f}")
    print(f"  mean reward, elapsed_steps = 0   : {bad:.4f}   ({bad/max(ok,1e-9):.2f}x inflated)")
else:
    print("  no counterfactual successes in this sample — try a later checkpoint")

### 1.3 Policy centering

$\sum_a \pi(a|s) A_{CF}(s,a) = 0$ must hold to floating-point precision, per
state. It is what makes the all-action gradient unbiased with respect to action
sampling; if it drifts, the loss acquires a state-dependent bias term and the
whole construction is unsound. The trainer logs this every update as
`cf_centering` and it should stay at zero for the entire run.

In [ ]:
# Recover the observation in each state the same way training does, then pi(.|s).
obs0 = np.stack([set_sim_state(oracle.env, s, elapsed_steps=int(s[-1])) for s in sims[:400]])
pi = probs_fn(obs0)
a_cf, q_cf = oracle.a_cf(sims[:400], pi, value_fn)
c = check_centering(a_cf, pi)
print(f"  max |sum_a pi*A_CF|   {c['max_abs']:.3e}")
print(f"  mean |sum_a pi*A_CF|  {c['mean_abs']:.3e}")
print(f"  -> {'OK' if c['ok'] else 'BIASED — STOP'}")

### 1.4 Is the landscape non-degenerate, and does it say sensible things?

Two different questions. *Non-degenerate*: do the actions actually differ, or is
$A_{CF}$ flat (which is what NB02 found at MountainCar's 30% checkpoint, where
the critic was a literal constant)? *Sensible*: in DoorKey, `drop` (4) and
`done` (6) can never help, so a correct oracle over a competent policy should
rank them below average almost everywhere. That is a probe, not a gate — but if
it fails on a trained policy, suspect the oracle before the environment.

In [ ]:
s_ = landscape_summary(a_cf, q_cf, pi, spread_threshold=cfg.oracle.spread_threshold,
                       useless_actions=(4, 6) if K == 7 else ())
names = ["left","right","forward","pickup","drop","toggle","done"][:K]
print(f"  mean |A_CF|                      {s_['mean_abs_a_cf']:.4f}")
print(f"  max  |A_CF|                      {s_['max_abs_a_cf']:.4f}")
print(f"  mean Q spread (max-min over a)   {s_['mean_q_spread']:.4f}")
print(f"  frac states with spread > {cfg.oracle.spread_threshold}    {s_['frac_states_with_spread']:.3f}")
print(f"  all finite                       {s_['finite']}")
print("\n  best action by A_CF, counts:")
for n, c_ in zip(names, s_["best_action_counts"]):
    print(f"    {n:8s} {c_:5d}")
if "useless_actions_negative_frac" in s_:
    print(f"\n  drop/done have A_CF < 0 in {s_['useless_actions_negative_frac']:.1%} of states")
    print("  (should be well above chance on a trained policy; ~50% means the oracle")
    print("   is reading an uninformative critic, not that the actions are useful)")

### 1.5 `fast` restore vs `exact`, and throughput

`cf_restore: "exact"` goes through `set_sim_state`, which calls `env.reset()`
before every restore — that reset is most of the cost. `"fast"` writes the grid
and agent state directly. It is only worth having if it is **bit-identical**,
so verify rather than assume, then read the timing and decide.

In [ ]:
eq = check_restore_equivalence(cfg.env.env_id, K, cfg.ppo.gamma, sims[:250],
                               ENV_KW, cfg.env.max_episode_steps)
print(f"  max reward diff   {eq['max_reward_diff']:.3e}")
print(f"  max obs diff      {eq['max_obs_diff']:.3e}")
print(f"  terminated agree  {eq['terminated_agreement']:.3f}")
print(f"  -> {'IDENTICAL — fast is safe to use' if eq['identical'] else 'DIFFERENT — keep exact'}\n")

batch = cfg.env.n_envs * cfg.ppo.n_steps
for kind in ("exact", "fast"):
    o = OnlineOracle(cfg.env.env_id, K, cfg.ppo.gamma, ENV_KW, cfg.env.max_episode_steps, restore=kind)
    probe = sims[:min(batch, len(sims))]
    t0 = time.time(); o.q_cf(probe, value_fn); dt = time.time() - t0
    sps = len(probe) / dt
    print(f"  {kind:5s}  {sps:6.0f} collected steps/s   "
          f"-> {cfg.ppo.total_timesteps/sps/60:5.1f} min/seed at {cfg.ppo.total_timesteps:,} frames")
    o.close()
oracle.close()

### Oracle verdict

In [ ]:
checks = {
    "restore/replay is exact":                    rep["exact"],
    "A_CF is policy-centred (max < 1e-4)":        c["ok"],
    "A_CF is finite":                             s_["finite"],
    f"actions differ in > {cfg.oracle.min_frac_states_with_spread:.0%} of states":
        s_["frac_states_with_spread"] > cfg.oracle.min_frac_states_with_spread,
    "fast and exact restore agree":               eq["identical"],
}
w = max(len(k) for k in checks)
for k, v in checks.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k:<{w}}")
print()
print("ORACLE:", "PASS — safe to train on" if all(checks.values())
      else "FAIL — fix before training; A_CF errors are invisible downstream")

---
# 2. Arm A — plain PPO (control)

In [ ]:
def train_arm(key):
    c = ARMS[key]
    done = all((seed_dir(c.run.run_name, s) / "scalars.csv").exists() for s in c.run.seeds)
    if done and not FORCE_RETRAIN:
        print(f"found existing run at {RUNS_DIR / c.run.run_name} — skipping")
        return None
    t0 = time.time()
    out = run_seeds(c)
    print(f"\n{key}: {(time.time()-t0)/60:.1f} min total")
    return out

if RUN_CONTROL:
    train_arm("gae")

---
# 3. Arm B — PPO-CF

The first rollout prints the oracle's replay and centering check again, this
time against the live policy inside the trainer. `cf_centering` in
`scalars.csv` should read 0 on every logged update.

In [ ]:
if RUN_CF:
    train_arm("cf")

---
# 4. Comparison

The metric is **sample efficiency**: frames to first reach
`success_rate_100 >= 0.9`. Final performance is not the axis — plain PPO already
reaches 1.00 on 5x5 — so a run that gets there sooner is the win condition.

In [ ]:
scal = {k: {s: read_scalars(seed_dir(c.run.run_name, s) / "scalars.csv")
            for s in c.run.seeds if (seed_dir(c.run.run_name, s) / "scalars.csv").exists()}
        for k, c in ARMS.items()}
scal = {k: v for k, v in scal.items() if v}

THRESH = 0.9
def frames_to(d, col="success_rate_100", thresh=THRESH):
    hit = d.loc[d[col] >= thresh, "global_step"]
    return int(hit.iloc[0]) if len(hit) else None

rows = []
for arm, per_seed in scal.items():
    for s, d in per_seed.items():
        rows.append({
            "arm": arm, "seed": s,
            f"frames_to_{THRESH:g}": frames_to(d),
            "final_success": round(float(d["success_rate_100"].iloc[-1]), 3),
            "final_door": round(float(d["door_rate_100"].iloc[-1]), 3),
            "best_success": round(float(d["success_rate_100"].max()), 3),
            "ev_median": round(float(d["explained_variance"].median()), 3),
            "kl_median": round(float(d["approx_kl"].median()), 4),
            "clipfrac_median": round(float(d["clipfrac"].median()), 3),
        })
cmp_df = pd.DataFrame(rows).sort_values(["arm", "seed"])
display(cmp_df)

print(f"\nmedian frames to success >= {THRESH:g}")
for arm, g in cmp_df.groupby("arm"):
    v = g[f"frames_to_{THRESH:g}"].dropna()
    print(f"  {arm:3s}  {('%.0f' % v.median()) if len(v) else 'never':>10s}"
          f"   ({len(v)}/{len(g)} seeds reached it)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = {"gae": "#3b6ea5", "cf": "#c4622d"}
panels = [("success_rate_100", "Success rate"), ("door_rate_100", "Door opened"),
          ("entropy", "Policy entropy")]
for ax, (col, title) in zip(axes, panels):
    for arm, per_seed in scal.items():
        for i, (s, d) in enumerate(per_seed.items()):
            ax.plot(d["global_step"], d[col], lw=1.4, alpha=0.85, color=colors[arm],
                    label=arm if i == 0 else None)
    ax.set_title(title, fontsize=10); ax.set_xlabel("environment steps")
    ax.spines[["top","right"]].set_visible(False); ax.grid(axis="y", alpha=0.25)
axes[0].axhline(THRESH, color="#888", ls=":", lw=1)
axes[0].set_ylim(-0.05, 1.05); axes[1].set_ylim(-0.05, 1.05)
axes[0].legend(frameon=False, fontsize=9)
fig.tight_layout(); savefig(fig, FIG / "arms.png"); plt.show()

### The confound to check before believing a win

The all-action gradient updates **all K action logits** per state, while GAE
updates only the sampled one. At the same learning rate that makes the CF arm
take *larger* policy steps — measured on a smoke run, `approx_kl` and `clipfrac`
came out roughly 3× higher for CF at the same frame count.

So if CF wins, the question is whether it won on counterfactual *information* or
just on a bigger effective step size. The panel below shows update size per arm.
If they are badly mismatched, rerun both with `ppo.target_kl: 0.03`, which caps
the per-update movement and makes the comparison KL-matched.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, col, title in [(axes[0], "approx_kl", "approx_kl (update size)"),
                       (axes[1], "clipfrac", "clipfrac"),
                       (axes[2], "adv_std_raw", "GAE advantage spread")]:
    for arm, per_seed in scal.items():
        for i, (s, d) in enumerate(per_seed.items()):
            ax.plot(d["global_step"], d[col], lw=1.3, alpha=0.85, color=colors[arm],
                    label=arm if i == 0 else None)
    ax.set_title(title, fontsize=10); ax.set_xlabel("environment steps")
    ax.spines[["top","right"]].set_visible(False); ax.grid(axis="y", alpha=0.25)
axes[2].set_yscale("log"); axes[0].legend(frameon=False, fontsize=9)
fig.tight_layout(); savefig(fig, FIG / "update_size.png"); plt.show()

if "cf" in scal:
    d = list(scal["cf"].values())[0]
    print(f"cf_centering over the whole run: max {d['cf_centering'].max():.3e} "
          f"(must be ~0; anything above 1e-4 invalidates the run)")
    print(f"cf_scale range: {d['cf_scale'].min():.4f} - {d['cf_scale'].max():.4f}")

---
## 5. Reading the result

| outcome | what it means | next |
|---|---|---|
| CF reaches 0.9 in clearly fewer frames, KL comparable | the counterfactual advantage helps | move up a rung; then NB05's learned $B(s)$ becomes worth building |
| CF wins but its `approx_kl` is much larger | probably step size, not information | rerun both with `ppo.target_kl: 0.03` |
| arms overlap | **5x5 is too easy to discriminate**, not "the oracle fails" | move up a rung — do not tune |
| CF is clearly worse | the oracle is exact, so suspect the critic: $A_{CF}$ inherits all of $V_\phi$'s error | check `ev_median` per arm |

The rung to move to is `doorkey6x6_cf` (750k, no warm start — a controlled
comparison must not inherit a policy trained under a different gradient), then
`doorkey8x8_cf` as a **probe** rather than a controlled comparison.

8x8 is where this matters most. Plain PPO failed it outright: 3M frames, success
never above 0.06, key rate 0.85 but door rate 0.10, entropy pinned at 1.89 of
log 7 = 1.946 — the policy was held at uniform because the sampled advantage
carried no consistent signal (`adv_std_raw` ~1e-3, EV oscillating around 0). If
action-sampling variance is what killed it, the all-action oracle is exactly the
intervention that should help. Watch the **door rate**: on 6x6, once the door
opened the agent essentially always finished, so the door is where 8x8 stalls.